# Bhu-Aadhaar 3D: GPU training (Kaggle)
Team Dimension Zero · SIH26011. Runs unattended: push it with `scripts/kaggle_train.sh` or open it on Kaggle with **GPU + Internet on**.

What it trains and saves to `/kaggle/working/models/`:
1. **Model 1 (U-Net)**: pretrain on African + South Asian drone imagery, register the Indian labels, fine-tune on India, test on held-out Indian scenes, export to ONNX for fast CPU use
2. **Model 1-lite (CPU LightGBM)** RGB-only, retrained on the full data (a fallback that needs no GPU at inference)

Set `QUICK = True` for a ~1-hour smoke run; `False` for the full run (~6-8 h on a P100/T4). Every step skips work already done, so you can re-run after a session timeout.

In [ ]:
QUICK = False
import os, glob, subprocess, sys
src = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)]
# work in /kaggle/temp: Kaggle saves everything under /kaggle/working as the run's output, and the
# image chips (many GB, thousands of files) made the output slow to save and impossible to fetch
WORK = '/kaggle/temp/ulpin3d-engine'
os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(WORK):
    assert src, 'add the ulpin3d-engine dataset to this notebook'
    subprocess.run(['cp', '-r', src[0], WORK], check=True)
os.chdir(WORK)
!pip install -q -e . 2>&1 | tail -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
SET = ""
if QUICK:
    SET = "--set train.epochs=3 train.steps_per_epoch=100"
    # fewer scenes: trim the chip budget
    !sed -i 's/south_asia: 60/south_asia: 15/; s/africa: 300/africa: 60/; s/india: 300/india: 120/' configs/data_footprint.yaml
!python -m ulpin3d.cli catalog --config configs/data_footprint.yaml | tail -6
!python -m ulpin3d.cli labels  --config configs/data_footprint.yaml --workers 8 | tail -3
!python -m ulpin3d.cli chips   --config configs/data_footprint.yaml --workers 8 | tail -8

## Stage 1: pretrain on look-alike regions

In [ ]:
if not os.path.exists('runs/footprint_stage1/best.pt'):
    !python -m ulpin3d.cli train-footprint --config configs/footprint_stage1_pretrain.yaml {SET}

## Register Indian / South Asian labels to the drone mosaics, rebuild their chips

In [ ]:
!python -m ulpin3d.cli register --config configs/data_footprint.yaml --checkpoint runs/footprint_stage1/best.pt | tail -15
!python -m ulpin3d.cli chips --config configs/data_footprint.yaml --sources oam --out data/footprint/chips_reg --workers 8 | tail -6

## Stage 2: fine-tune on India

In [ ]:
if not os.path.exists('runs/footprint_stage2_india/best.pt'):
    !python -m ulpin3d.cli train-footprint --config configs/footprint_stage2_india.yaml --set data.chips_root=data/footprint/chips_reg {SET}

## Test on held-out Indian scenes

In [ ]:
!python -m ulpin3d.cli eval-footprint --checkpoint runs/footprint_stage1/best.pt       --chips data/footprint/chips_reg --split test --out reports/stage1_india_test --no-tta | tail -8
!python -m ulpin3d.cli eval-footprint --checkpoint runs/footprint_stage2_india/best.pt --chips data/footprint/chips_reg --split test --out reports/stage2_india_test --no-tta | tail -8

## Model 1-lite (CPU) on the full data, for comparison and as a no-GPU fallback

In [ ]:
!python -m ulpin3d.cli train-footprint-lite --chips data/footprint/chips_reg --out models/footprint_lite_rgb_full --height no --splits train,val
!python -m ulpin3d.cli eval-footprint-lite --model models/footprint_lite_rgb_full --chips data/footprint/chips_reg --split test --out reports/lite_rgb_full_india_test | tail -4

## Export for fast CPU inference and save everything

In [ ]:
!python -m ulpin3d.cli export-onnx --checkpoint runs/footprint_stage2_india/best.pt --out models/footprint_india.onnx
!mkdir -p /kaggle/working/models
!cp runs/footprint_stage2_india/best.pt /kaggle/working/models/footprint_india.pt
!cp runs/footprint_stage1/best.pt /kaggle/working/models/footprint_stage1.pt
!cp models/footprint_india.onnx* /kaggle/working/models/
!cp -r models/footprint_lite_rgb_full reports /kaggle/working/models/
!ls -la /kaggle/working/models